# FMCG Sales Intelligence — Integration Example

This tutorial demonstrates supported integration boundaries. It performs no training and never writes canonical artifacts.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'examples':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'examples'))
from backend_integration import run_direct_usage  # noqa: E402

usage = run_direct_usage()
'Frozen usage results loaded'

## Available capabilities

The domain pack declares available canonical contracts. Capability evaluation prevents a missing contract from being treated as a usable model input.

In [2]:
pd.DataFrame(usage['domain_capabilities'])[['capability', 'status', 'missing_or_invalid']]

## External data → canonical contract

`FileAdapter` applies the selected domain-pack mapping. `ContractRegistry` then checks required columns, nulls and business-key duplicates. A structural PASS is necessary but does not replace semantic data-quality review.

In [3]:
usage['external_data_contract']['normalized_rows'], usage['external_data_contract']['validation']

## Forecasting — ONLINE / BATCH SERVING

`ForecastService` accepts the frozen V1 feature contract and returns the predicted realized units in `(t,t+7d]`. Raw `sales_daily` rows must first pass feature construction.

In [4]:
pd.Series(usage['forecasting']['input'], name='forecast input')

In [5]:
usage['forecasting']['result']

## Stockout Classification — ONLINE / BATCH SERVING

`StockoutService` returns a probability and the frozen validation-derived decision threshold. The score is not claimed to be strongly calibrated.

In [6]:
usage['stockout_classification']

## Store Segmentation — EXPLORATORY FROZEN ASSIGNMENT

The service applies the existing KMeans state without fitting. Its cluster is a pseudo-label with limited taxonomy stability and mandatory review.

In [7]:
usage['segmentation_membership']

## Anomaly Detection — OFFLINE HUMAN-REVIEW ANALYTICS

The product interface reads frozen review candidates. It does not expose a fake online anomaly predictor and does not treat candidates as confirmed events.

In [8]:
anomalies = usage['offline_frozen_outputs']['anomalies']
pd.DataFrame(anomalies['items'])[['event_date', 'store_id', 'sku_id', 'iforest_anomaly_score', 'review_status']]

## Market Basket Analysis — OFFLINE ASSOCIATION ANALYTICS

Rules describe co-occurrence. They are not recommendations and do not establish causality.

In [9]:
rules = usage['offline_frozen_outputs']['basket-rules']
pd.DataFrame(rules['items'])[['antecedent', 'consequent', 'support', 'confidence', 'lift']]

## Promotion Performance — OFFLINE DESCRIPTIVE ANALYTICS

The frozen output is a PRE/DURING/POST comparison. It is explicitly non-causal and does not claim incrementality or ROI.

In [10]:
promotions = usage['offline_frozen_outputs']['promotions']
pd.DataFrame(promotions['items'])[['promotion_id', 'pre_units_per_day', 'during_units_per_day', 'post_units_per_day', 'analysis_semantics']]

## Stockout Survival — OFFLINE ANALYTICAL SCORING

The scientific API applies the frozen Cox PH bundle to an eligible feature row. Survival probabilities are horizon-specific event-free probabilities, not classification probabilities. No REST endpoint is declared.

In [11]:
usage['stockout_survival']

## Integration summary

- **Serving:** forecasting, stockout classification and exploratory segment membership use frozen artifacts.
- **Offline scoring:** survival has a Python scientific interface but no REST route.
- **Frozen analytical reads:** anomaly candidates, basket rules and promotion summaries are available in Python and REST.
- **External onboarding:** map source columns, validate canonical contracts, evaluate capability availability, then run the reviewed task-dataset pipeline.
- **Not performed here:** model fitting, canonical artifact generation, DVC operations or MLflow runtime lookup.